## Pré-processamento e base analítica

### 0. Imports

In [1]:
import joblib
import numpy as np
import pandas as pd
from google.cloud import bigquery
from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

### 1. Parâmetros

In [2]:
par_source_project = "tech-challenge-fase-2-505123"
par_gold = f"{par_source_project}.gold"
par_bd = "basedosdados"

par_anos_alvo = [2023, 2024, 2025]
par_ano_principal = 2024
par_ano_escoragem = 2025
par_ano_inep = 2022
par_rede_estadual = "2"
par_rede_municipal = "3"
par_redes = [par_rede_estadual, par_rede_municipal]
par_k_suavizacao = 30
par_k_vizinhos = 8
par_peso_maximo = 5
par_seed = 42
par_prop_teste = .15
par_prop_validacao = .15
par_saida = Path("../data/processed")

client = bigquery.Client(project=par_source_project)
pd.set_option("display.max_columns", 60, "display.max_rows", 80, "display.float_format", "{:,.3f}".format)

### 2. Funções Auxiliares

In [3]:
def ler_tabela(tabela: str, colunas: list | None = None) -> pd.DataFrame:
    t = client.get_table(tabela)
    if t.table_type == "VIEW":
        return client.query(f"select {', '.join(colunas) if colunas else '*'} from `{tabela}`").to_dataframe()
    campos = [c for c in t.schema if c.name in colunas] if colunas else None
    return client.list_rows(t, selected_fields=campos).to_dataframe()


def para_numero(s):
    return pd.to_numeric(s, errors="coerce")


def divisao_segura(a, b):
    return (a / b.where(b != 0)).astype("float64")


def media_vizinhos(valores, coords, k):
    valores, coords = np.asarray(valores, dtype=float), np.asarray(coords, dtype=float)
    validos = np.flatnonzero(~np.isnan(valores) & ~np.isnan(coords).any(axis=1))
    com_coord = ~np.isnan(coords).any(axis=1)
    vizinhos = cKDTree(coords[validos]).query(coords[com_coord], k=k + 1)[1]
    origem = validos[vizinhos]
    proprio = origem == np.flatnonzero(com_coord)[:, None]
    proprio[~proprio.any(axis=1), -1] = True
    resultado = np.full(len(valores), np.nan)
    resultado[com_coord] = np.ma.array(valores[origem], mask=proprio).mean(axis=1).filled(np.nan)
    return resultado


def suavizar(valor, n, referencia, k):
    return (n * valor + k * referencia) / (n + k)


def reduzir_numericas(df):
    colunas = df.select_dtypes("float64").columns
    return df.astype({c: "float32" for c in colunas})


def construir_preprocessador(numericas, numericas_log, categoricas):
    imputacao = SimpleImputer(strategy="median")
    return ColumnTransformer([
        ("num", Pipeline([("imputacao", imputacao), ("escala", StandardScaler())]), numericas),
        ("log", Pipeline([("imputacao", imputacao), ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                          ("escala", StandardScaler())]), numericas_log),
        ("ausente", MissingIndicator(features="missing-only", error_on_new=False), numericas + numericas_log),
        ("cat", Pipeline([("imputacao", SimpleImputer(strategy="constant", fill_value="ausente")),
                          ("encoding", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categoricas),
    ])


def auc_univariada(df, colunas, alvo, peso=None):
    linhas = []
    for c in colunas:
        d = df[df[c].notna()]
        valor = roc_auc_score(d[alvo], d[c], sample_weight=None if peso is None else d[peso])
        linhas.append({"feature": c, "auc": valor, "forca": abs(valor - .5) + .5, "pct_nulos": 100 * df[c].isna().mean()})
    return pd.DataFrame(linhas).sort_values("forca", ascending=False).reset_index(drop=True)

### 3. Leitura de dados

In [4]:
df_scr_municipio = ler_tabela(f"{par_gold}.dim_municipio")
df_scr_rede = ler_tabela(f"{par_gold}.dim_rede")
df_scr_meta = ler_tabela(f"{par_gold}.fato_meta")
df_scr_indicador = ler_tabela(f"{par_gold}.fato_indicador_municipio")
df_scr_resultados = ler_tabela(f"{par_gold}.fato_resultados")

In [5]:
df_scr_fato_aluno = ler_tabela(f"{par_gold}.fato_aluno",
                               ["ano", "id_escola", "id_municipio", "rede_id", "proficiencia", "presenca", "alfabetizado_id", "peso_aluno"])
df_scr_fato_aluno["alunos_previstos_escola"] = (df_scr_fato_aluno.groupby(["ano", "id_escola"], dropna=False)["ano"]
                                                .transform("size").astype("Int64"))
df_scr_aluno = (df_scr_fato_aluno[df_scr_fato_aluno["proficiencia"].notna() & df_scr_fato_aluno["rede_id"].isin(par_redes)]
                .rename(columns={"alfabetizado_id": "alfabetizado"})
                [["ano", "id_escola", "id_municipio", "rede_id", "alfabetizado", "peso_aluno", "alunos_previstos_escola"]]
                .sort_values(["ano", "id_municipio", "rede_id", "id_escola"], kind="stable").reset_index(drop=True))
df_scr_aluno.shape

(3354637, 7)

In [6]:
df_participacao = df_scr_fato_aluno[df_scr_fato_aluno["rede_id"].isin(par_redes)]
df_scr_participacao = (df_participacao.assign(avaliado=df_participacao["proficiencia"].notna())
                       .groupby(["ano", "id_municipio", "rede_id"], as_index=False, dropna=False)
                       .agg(n_total=("ano", "size"), n_presentes=("presenca", "sum"), n_avaliados=("avaliado", "sum"))
                       .astype({c: "Int64" for c in ["n_total", "n_presentes", "n_avaliados"]}))

**Fontes externas: cada ano-alvo recebe apenas dados de anos anteriores**

In [7]:
df_scr_atlas = (ler_tabela(f"{par_bd}.mundo_onu_adh.municipio",
                           ["ano", "id_municipio", "idhm", "idhm_e", "renda_pc", "indice_gini", "prop_pobreza_criancas", "expectativa_anos_estudo"])
                .query("ano == 2010").drop(columns="ano").reset_index(drop=True))

In [8]:
df_scr_ibge_populacao = ler_tabela(f"{par_bd}.br_ibge_populacao.municipio", ["ano", "id_municipio", "populacao"])
df_scr_populacao = (df_scr_ibge_populacao[df_scr_ibge_populacao["ano"].between(2022, 2024)]
                    .assign(ano_alvo=lambda d: d["ano"] + 1).rename(columns={"ano": "ano_ref"})
                    [["ano_alvo", "ano_ref", "id_municipio", "populacao"]].reset_index(drop=True))

In [9]:
df_scr_ibge_pib = ler_tabela(f"{par_bd}.br_ibge_pib.municipio", ["ano", "id_municipio", "pib"])
df_scr_pib = (df_scr_ibge_pib[df_scr_ibge_pib["ano"].between(2021, 2023)]
              .merge(df_scr_ibge_populacao, on=["ano", "id_municipio"])
              .assign(ano_alvo=lambda d: d["ano"] + 2, pib_per_capita=lambda d: divisao_segura(d["pib"], d["populacao"]))
              .rename(columns={"ano": "ano_ref"})
              [["ano_alvo", "ano_ref", "id_municipio", "pib_per_capita"]])

In [10]:
par_matricula, par_turma, par_docente = [f"quantidade_{c}_fundamental_anos_iniciais" for c in ["matricula", "turma", "docente"]]
df_scr_censo_escola = ler_tabela(f"{par_bd}.br_inep_censo_escolar.escola",
                                 ["ano", "id_municipio", "rede", "tipo_situacao_funcionamento", "etapa_ensino_fundamental_anos_iniciais",
                                  "tipo_localizacao", "biblioteca_sala_leitura", "esgoto_rede_publica", par_matricula, par_turma, par_docente])
df_censo = df_scr_censo_escola.assign(rede_id=para_numero(df_scr_censo_escola["rede"]).astype("Int64").astype("str"))
df_censo = df_censo[df_censo["ano"].between(2022, 2024) & df_censo["rede_id"].isin(par_redes)
                    & (para_numero(df_censo["tipo_situacao_funcionamento"]) == 1)
                    & (para_numero(df_censo["etapa_ensino_fundamental_anos_iniciais"]) == 1)]
df_censo = df_censo.assign(ano_alvo=df_censo["ano"] + 1, rural=(para_numero(df_censo["tipo_localizacao"]) == 2).astype(int))
g_censo = df_censo.rename(columns={"ano": "ano_ref"}).groupby(["ano_alvo", "ano_ref", "id_municipio", "rede_id"], dropna=False)
soma_censo = g_censo[[par_matricula, par_turma, par_docente]].sum(min_count=1)
df_scr_censo = g_censo.agg(escolas_anos_iniciais=("rural", "size")).astype("Int64")
df_scr_censo["matriculas_anos_iniciais"] = soma_censo[par_matricula]
df_scr_censo["pct_escolas_rurais"] = 100 * g_censo["rural"].mean()
df_scr_censo["pct_biblioteca"] = 100 * g_censo["biblioteca_sala_leitura"].mean().astype("float64")
df_scr_censo["pct_esgoto_rede"] = 100 * g_censo["esgoto_rede_publica"].mean().astype("float64")
df_scr_censo["alunos_por_turma"] = divisao_segura(soma_censo[par_matricula], soma_censo[par_turma])
df_scr_censo["alunos_por_docente"] = divisao_segura(soma_censo[par_matricula], soma_censo[par_docente])
df_scr_censo = df_scr_censo.reset_index()

In [11]:
par_colunas_inep = {"atu_ef_2_ano": "alunos_turma_2ano", "tdi_ef_anos_iniciais": "distorcao_idade_serie",
                    "taxa_aprovacao_ef_anos_iniciais": "taxa_aprovacao", "taxa_abandono_ef_anos_iniciais": "taxa_abandono",
                    "dsu_ef_anos_iniciais": "pct_docente_superior", "afd_ef_anos_iniciais_grupo_1": "pct_docente_adequado",
                    "ird_baixa_regularidade": "pct_docente_baixa_regularidade"}
df_inep = ler_tabela(f"{par_bd}.br_inep_indicadores_educacionais.municipio", ["ano", "id_municipio", "localizacao", "rede", *par_colunas_inep])
df_inep = df_inep[(df_inep["ano"] == par_ano_inep) & (df_inep["localizacao"].str.lower() == "total")
                  & df_inep["rede"].str.lower().isin(["estadual", "municipal"])]
df_scr_inep = (df_inep.assign(rede_id=np.where(df_inep["rede"].str.lower() == "estadual", par_rede_estadual, par_rede_municipal))
               [["id_municipio", "rede_id", *par_colunas_inep]].rename(columns=par_colunas_inep).reset_index(drop=True))
df_scr_inep[list(par_colunas_inep.values())] = df_scr_inep[list(par_colunas_inep.values())].apply(para_numero).astype("float64")

In [12]:
def investimento_fundeb(tabela, chave, indicador):
    d = ler_tabela(f"{par_bd}.br_fnde_fundeb.{tabela}", ["ano", "bimestre", chave, "id_indicador", "valor_real"])
    d = d[d["ano"].between(2022, 2024) & (d["bimestre"] == 6) & (d["id_indicador"].astype(str) == indicador)]
    return (d.assign(ano_alvo=d["ano"] + 1).rename(columns={"ano": "ano_ref"})
            .groupby(["ano_alvo", "ano_ref", chave], as_index=False, dropna=False)
            .agg(invest_aluno_fundamental=("valor_real", "max")))


df_scr_fundeb_municipal = investimento_fundeb("indicador_municipal", "id_municipio", "45")
df_scr_fundeb_estadual = investimento_fundeb("indicador_estadual", "sigla_uf", "66")

In [13]:
df_scr_territorio = ler_tabela(f"{par_bd}.br_bd_diretorios_brasil.municipio",
                               ["id_municipio", "nome_mesorregiao", "capital_uf", "amazonia_legal", "centroide"])
df_scr_territorio[["capital_uf", "amazonia_legal"]] = df_scr_territorio[["capital_uf", "amazonia_legal"]].astype("str")
df_scr_territorio[["lon", "lat"]] = df_scr_territorio["centroide"].astype(str).str.extract(r"POINT\((\S+) (\S+)\)").astype(float)
df_scr_territorio = df_scr_territorio.drop(columns="centroide")

### 4. Contexto município × rede × ano-alvo

In [14]:
df_contexto = (pd.MultiIndex.from_product([par_anos_alvo, df_scr_municipio["id_municipio"], par_redes],
                                          names=["ano_alvo", "id_municipio", "rede_id"]).to_frame(index=False)
               .merge(df_scr_municipio[["id_municipio", "nome_municipio", "sigla_uf", "nome_regiao"]], on="id_municipio")
               .merge(df_scr_territorio, on="id_municipio", how="left")
               .merge(df_scr_rede, on="rede_id", how="left"))
df_contexto.shape

(33306, 12)

**Socioeconômicas e educacionais**

In [15]:
df_contexto = (df_contexto
               .merge(df_scr_atlas, on="id_municipio", how="left")
               .merge(df_scr_populacao.drop(columns="ano_ref"), on=["ano_alvo", "id_municipio"], how="left")
               .merge(df_scr_pib.drop(columns="ano_ref"), on=["ano_alvo", "id_municipio"], how="left")
               .merge(df_scr_censo.drop(columns="ano_ref"), on=["ano_alvo", "id_municipio", "rede_id"], how="left")
               .merge(df_scr_inep, on=["id_municipio", "rede_id"], how="left"))

In [16]:
df_contexto = (df_contexto
               .merge(df_scr_fundeb_municipal.drop(columns="ano_ref").rename(columns={"invest_aluno_fundamental": "invest_municipal"}),
                      on=["ano_alvo", "id_municipio"], how="left")
               .merge(df_scr_fundeb_estadual.drop(columns="ano_ref").rename(columns={"invest_aluno_fundamental": "invest_estadual"}),
                      on=["ano_alvo", "sigla_uf"], how="left"))
df_contexto["invest_aluno_fundamental"] = np.where(df_contexto["rede_id"] == par_rede_municipal,
                                                   df_contexto["invest_municipal"], df_contexto["invest_estadual"])
df_contexto["invest_aluno_fundamental"] = df_contexto["invest_aluno_fundamental"].where(df_contexto["invest_aluno_fundamental"] > 0)
df_contexto = df_contexto.drop(columns=["invest_municipal", "invest_estadual"])

**Resultados do ano anterior**

In [17]:
df_lag_indicador = (df_scr_indicador[df_scr_indicador["rede_id"].isin(par_redes)]
                    .assign(ano_alvo=lambda d: d["ano"] + 1)
                    .rename(columns={"taxa_alfabetizacao": "taxa_lag", "media_portugues": "media_portugues_lag"})
                    [["ano_alvo", "id_municipio", "rede_id", "taxa_lag", "media_portugues_lag"]])
df_lag_participacao = df_scr_participacao.assign(ano_alvo=lambda d: d["ano"] + 1,
                                                 taxa_presenca_lag=lambda d: 100 * d["n_presentes"] / d["n_total"],
                                                 n_avaliados_lag=lambda d: d["n_avaliados"].astype(float))
df_contexto = (df_contexto
               .merge(df_lag_indicador, on=["ano_alvo", "id_municipio", "rede_id"], how="left")
               .merge(df_lag_participacao[["ano_alvo", "id_municipio", "rede_id", "taxa_presenca_lag", "n_avaliados_lag"]],
                      on=["ano_alvo", "id_municipio", "rede_id"], how="left"))

In [18]:
df_contexto["taxa_lag_uf"] = df_contexto.groupby(["ano_alvo", "rede_id", "sigla_uf"])["taxa_lag"].transform("mean")
df_contexto["taxa_lag_suavizada"] = suavizar(df_contexto["taxa_lag"], df_contexto["n_avaliados_lag"],
                                             df_contexto["taxa_lag_uf"], par_k_suavizacao).fillna(df_contexto["taxa_lag"])

In [19]:
df_contexto["taxa_lag_vizinhos"] = np.nan
for (ano, rede), d in df_contexto.groupby(["ano_alvo", "rede_id"]):
    if d["taxa_lag"].notna().any():
        df_contexto.loc[d.index, "taxa_lag_vizinhos"] = media_vizinhos(d["taxa_lag"], d[["lon", "lat"]], par_k_vizinhos)
df_contexto[["taxa_lag", "taxa_lag_suavizada", "taxa_lag_uf", "taxa_lag_vizinhos"]].describe().T

,count,mean,std,min,25%,50%,75%,max
taxa_lag,"13,131.000",61.846,19.644,2.120,47.820,62.520,76.795,100.000
taxa_lag_suavizada,"13,131.000",61.482,16.478,14.182,49.764,62.716,74.244,100.000
taxa_lag_uf,"21,787.000",62.004,13.877,28.718,52.971,63.297,73.950,90.310
taxa_lag_vizinhos,"22,204.000",62.600,13.823,18.988,53.809,63.756,72.940,99.175


**Metas (somente base municipal)**

In [20]:
df_meta_municipio = (df_scr_meta[df_scr_meta["nivel_geografico"] == "municipio"]
                     .sort_values("ano_referencia").drop_duplicates(["local_id", "ano_meta"], keep="last")
                     .rename(columns={"local_id": "id_municipio", "ano_meta": "ano_alvo", "valor_meta": "meta_ano"})
                     [["ano_alvo", "id_municipio", "meta_ano"]])
df_contexto = df_contexto.merge(df_meta_municipio.assign(rede_id=par_rede_municipal), on=["ano_alvo", "id_municipio", "rede_id"], how="left")
df_contexto["esforco_meta"] = df_contexto["meta_ano"] - df_contexto["taxa_lag"]
df_contexto = reduzir_numericas(df_contexto)
df_contexto.shape

(33306, 44)

### 5. Features

In [21]:
par_familias = {
    "rede": ["rede"],
    "territorio_categorico": ["sigla_uf", "nome_regiao", "capital_uf", "amazonia_legal"],
    "territorio_numerico": ["lat", "lon"],
    "historico": ["taxa_lag", "taxa_lag_suavizada", "media_portugues_lag", "taxa_presenca_lag", "n_avaliados_lag",
                  "taxa_lag_uf", "taxa_lag_vizinhos"],
    "socioeconomico": ["idhm", "idhm_e", "renda_pc", "indice_gini", "prop_pobreza_criancas", "expectativa_anos_estudo",
                       "populacao", "pib_per_capita"],
    "infraestrutura": ["escolas_anos_iniciais", "matriculas_anos_iniciais", "pct_escolas_rurais", "pct_biblioteca", "pct_esgoto_rede"],
    "docentes_turmas": ["alunos_por_turma", "alunos_por_docente", "alunos_turma_2ano", "pct_docente_superior",
                        "pct_docente_adequado", "pct_docente_baixa_regularidade"],
    "fluxo_escolar": ["distorcao_idade_serie", "taxa_aprovacao", "taxa_abandono"],
    "financiamento": ["invest_aluno_fundamental"],
    "escola": ["alunos_previstos_escola"],
    "metas": ["meta_ano", "esforco_meta"],
}
par_familias_metadados = pd.DataFrame([
    ("rede", "Gold", "ano-alvo", "seção 8", "baixo"),
    ("territorio_categorico", "Gold / Base dos Dados", "fixo", "seções 8, 12 e 14", "pode mascarar políticas estaduais; avaliado com e sem território"),
    ("territorio_numerico", "Base dos Dados", "fixo", "seção 8", "baixo"),
    ("historico", "Gold", "ano-alvo - 1", "seções 7, 8, 9 e 14", "só existe no alvo 2024 em diante"),
    ("socioeconomico", "Atlas 2010 / IBGE", "2010; até ano-alvo - 1", "seções 11 e 12", "Atlas defasado; alta redundância"),
    ("infraestrutura", "Censo Escolar", "ano-alvo - 1", "seção 11", "baixo"),
    ("docentes_turmas", "Censo Escolar / INEP", "2022; ano-alvo - 1", "seção 11", "baixo"),
    ("fluxo_escolar", "INEP", "2022", "seções 11 e 12", "pode refletir desempenho anterior"),
    ("financiamento", "Fundeb / SIOPE", "ano-alvo - 1", "seção 11", "associação fraca; municipal e estadual diferem"),
    ("escola", "Gold", "ano-alvo", "seção 8", "supõe lista de alunos prévia à prova"),
    ("metas", "Gold", "definida antes do ano-alvo", "seção 10", "só no modelo municipal"),
], columns=["familia", "fonte", "referencia", "evidencia_eda", "risco"])
par_numericas_log = ["n_avaliados_lag", "renda_pc", "populacao", "pib_per_capita", "escolas_anos_iniciais", "matriculas_anos_iniciais",
                     "alunos_por_turma", "alunos_por_docente", "invest_aluno_fundamental", "alunos_previstos_escola"]
par_categoricas = par_familias["rede"] + par_familias["territorio_categorico"]
par_features_municipio = [f for fam, cols in par_familias.items() if fam not in ["rede", "escola"] for f in cols]
par_features_aluno = [f for fam, cols in par_familias.items() if fam != "metas" for f in cols]

df_catalogo = (pd.DataFrame([{"feature": f, "familia": fam} for fam, cols in par_familias.items() for f in cols])
               .merge(par_familias_metadados, on="familia"))
df_catalogo["tipo"] = np.where(df_catalogo["feature"].isin(par_categoricas), "categórica", "numérica")
df_catalogo["transformacao"] = np.select(
    [df_catalogo["feature"].isin(par_categoricas), df_catalogo["feature"].isin(par_numericas_log)],
    ["ausente + one-hot", "mediana + log1p + padronização"], "mediana + padronização")
df_catalogo["modelo_aluno"] = df_catalogo["feature"].isin(par_features_aluno)
df_catalogo["modelo_municipio"] = df_catalogo["feature"].isin(par_features_municipio)
df_catalogo

,feature,familia,fonte,referencia,evidencia_eda,risco,tipo,transformacao,modelo_aluno,modelo_municipio
0,rede,rede,Gold,ano-alvo,seção 8,baixo,categórica,ausente + one-hot,True,False
1,sigla_uf,territorio_categorico,Gold / Base dos Dados,fixo,"seções 8, 12 e 14",pode mascarar políticas estaduais; avaliado co...,categórica,ausente + one-hot,True,True
2,nome_regiao,territorio_categorico,Gold / Base dos Dados,fixo,"seções 8, 12 e 14",pode mascarar políticas estaduais; avaliado co...,categórica,ausente + one-hot,True,True
3,capital_uf,territorio_categorico,Gold / Base dos Dados,fixo,"seções 8, 12 e 14",pode mascarar políticas estaduais; avaliado co...,categórica,ausente + one-hot,True,True
4,amazonia_legal,territorio_categorico,Gold / Base dos Dados,fixo,"seções 8, 12 e 14",pode mascarar políticas estaduais; avaliado co...,categórica,ausente + one-hot,True,True
5,lat,territorio_numerico,Base dos Dados,fixo,seção 8,baixo,numérica,mediana + padronização,True,True
6,lon,territorio_numerico,Base dos Dados,fixo,seção 8,baixo,numérica,mediana + padronização,True,True
7,taxa_lag,historico,Gold,ano-alvo - 1,"seções 7, 8, 9 e 14",só existe no alvo 2024 em diante,numérica,mediana + padronização,True,True
8,taxa_lag_suavizada,historico,Gold,ano-alvo - 1,"seções 7, 8, 9 e 14",só existe no alvo 2024 em diante,numérica,mediana + padronização,True,True
9,media_portugues_lag,historico,Gold,ano-alvo - 1,"seções 7, 8, 9 e 14",só existe no alvo 2024 em diante,numérica,mediana + padronização,True,True


**Fora do conjunto de teste**

In [22]:
df_excluidas = pd.DataFrame([
    ("proficiencia", "vazamento", "define o alvo (EDA, seção 13)"),
    ("media_portugues, proporcao_aluno_nivel_*, taxa_alfabetizacao do mesmo ano", "vazamento", "agregam o alvo (EDA, seção 13)"),
    ("presenca, preenchimento_caderno e IDs", "vazamento", "conhecidas só após a prova (EDA, seção 13)"),
    ("peso_aluno", "vazamento", "criado após a prova; usado só como sample_weight (EDA, seções 6 e 13)"),
    ("gap_meta_ano, gap_ate_2030, atingiu_meta_ano", "vazamento", "revelam o alvo municipal (EDA, seção 13)"),
    ("médias de escola ou município do mesmo ano", "vazamento", "incluem o alvo (EDA, seção 13)"),
    ("id_aluno", "identificação", "identificador individual, sem conteúdo preditivo e sem uso fora do treino"),
    ("id_escola", "não estável", "recodificado a cada ano e sem correspondência com o Censo (EDA, seção 5)"),
    ("id_municipio", "generalização", "não se aplica a municípios fora do treino; o histórico já cobre esse sinal"),
    ("nome_mesorregiao", "validação", "é a chave de separação entre treino e teste (EDA, seção 8)"),
    ("ano", "generalização", "não se projeta para anos futuros; usado só para filtro"),
    ("serie", "sem variância", "o indicador avalia apenas o 2º ano"),
    ("caderno", "vazamento", "versão da prova, definida só na aplicação"),
    ("pct_internet_alunos, horas_aula_2ano, despesa_professor_aluno", "sem sinal", "associação próxima de zero (EDA, seção 11)"),
    ("proporções por nível do ano anterior", "indisponível", "não existem em 2023 (EDA, seção 6)"),
], columns=["coluna", "motivo", "justificativa"])
df_excluidas

,coluna,motivo,justificativa
0,proficiencia,vazamento,"define o alvo (EDA, seção 13)"
1,"media_portugues, proporcao_aluno_nivel_*, taxa...",vazamento,"agregam o alvo (EDA, seção 13)"
2,"presenca, preenchimento_caderno e IDs",vazamento,"conhecidas só após a prova (EDA, seção 13)"
3,peso_aluno,vazamento,criado após a prova; usado só como sample_weig...
4,"gap_meta_ano, gap_ate_2030, atingiu_meta_ano",vazamento,"revelam o alvo municipal (EDA, seção 13)"
5,médias de escola ou município do mesmo ano,vazamento,"incluem o alvo (EDA, seção 13)"
6,id_aluno,identificação,"identificador individual, sem conteúdo prediti..."
7,id_escola,não estável,recodificado a cada ano e sem correspondência ...
8,id_municipio,generalização,não se aplica a municípios fora do treino; o h...
9,nome_mesorregiao,validação,é a chave de separação entre treino e teste (E...


**Decisão:** 38 features no modelo de aluno e 38 no municipal (36 em comum, entre as 40 do catálogo), organizadas por família para testes de inclusão e exclusão. Redundantes como IDHM, renda e pobreza infantil foram mantidas: a regularização e os testes por família, nos notebooks 3 e 4, definem o peso de cada uma. Dentro de um mesmo município, rede e ano, só `alunos_previstos_escola` varia entre alunos, então as demais descrevem o contexto, não a criança.

### 6. Bases analíticas

**Base de alunos: somente avaliados das redes Estadual e Municipal**

In [23]:
df_aluno = (df_scr_aluno
            .rename(columns={"ano": "ano_alvo"})
            .merge(df_contexto.drop(columns=par_familias["metas"]), on=["ano_alvo", "id_municipio", "rede_id"], how="left"))
df_aluno["alunos_previstos_escola"] = df_aluno["alunos_previstos_escola"].astype("float32")
df_aluno["alfabetizado"] = df_aluno["alfabetizado"].astype("int8")
df_aluno["peso_amostral"] = df_aluno["peso_aluno"].clip(upper=par_peso_maximo).astype("float32")
df_aluno = (df_aluno.drop(columns="peso_aluno")
            .sort_values(["ano_alvo", "id_municipio", "rede_id", "id_escola", "alfabetizado", "peso_amostral"], kind="stable").reset_index(drop=True))
print(df_aluno.shape, f"{df_aluno.memory_usage(deep=True).sum() / 1e9:.1f} GB")
df_aluno.groupby(["ano_alvo", "rede"]).agg(alunos=("alfabetizado", "size"), pct_alfabetizado=("alfabetizado", lambda s: 100 * s.mean()))

(3354637, 46) 1.0 GB


alunos  pct_alfabetizado
ano_alvo rede                                
2023     Estadual    131522            61.330
         Municipal  1371287            58.103
2024     Estadual    241074            62.537
         Municipal  1610754            59.372

**Base municipal: rede Municipal em 2024, com meta definida**

In [24]:
df_alvo_municipio = df_scr_resultados[(df_scr_resultados["rede_id"] == par_rede_municipal) & (df_scr_resultados["ano"] == par_ano_principal)]
df_municipio = (df_contexto[(df_contexto["ano_alvo"] == par_ano_principal) & (df_contexto["rede_id"] == par_rede_municipal)]
                .merge(df_alvo_municipio[["id_municipio", "taxa_alfabetizacao", "atingiu_meta_ano"]], on="id_municipio"))
print("Sem meta, fora do alvo:", df_municipio["meta_ano"].isna().sum())
df_municipio = df_municipio[df_municipio["meta_ano"].notna()].copy()
df_municipio["atingiu_meta"] = df_municipio["atingiu_meta_ano"].astype(int)
df_municipio = df_municipio.drop(columns="atingiu_meta_ano")
print(df_municipio.shape, "| % atingiu:", round(100 * df_municipio["atingiu_meta"].mean(), 1))

Sem meta, fora do alvo: 216
(5232, 46) | % atingiu: 53.3


**Base de escoragem: metas de 2025, a partir dos resultados de 2024**

In [25]:
df_escoragem = df_contexto[(df_contexto["ano_alvo"] == par_ano_escoragem) & (df_contexto["rede_id"] == par_rede_municipal)
                           & df_contexto["meta_ano"].notna() & df_contexto["taxa_lag"].notna()].copy()
df_escoragem[["ano_alvo", "taxa_lag", "meta_ano", "esforco_meta"]].describe().T

,count,mean,std,min,25%,50%,75%,max
ano_alvo,"5,352.000","2,025.000",0.000,"2,025.000","2,025.000","2,025.000","2,025.000","2,025.000"
taxa_lag,"5,352.000",63.042,19.252,4.400,48.730,64.250,77.712,100.000
meta_ano,"5,352.000",65.245,12.633,14.050,57.108,66.940,76.510,80.000
esforco_meta,"5,352.000",2.204,14.789,-76.280,-7.633,2.245,11.348,68.900


### 7. Data leakage

**Separação por mesorregião, para evitar vizinhos em treino e teste**

In [26]:
df_grupos = (df_aluno[df_aluno["ano_alvo"] == par_ano_principal].groupby(["sigla_uf", "nome_mesorregiao"], as_index=False)
             .agg(alunos=("alfabetizado", "size"))
             .sample(frac=1, random_state=par_seed).reset_index(drop=True))
par_metas_split = {"treino": 1 - par_prop_teste - par_prop_validacao, "validacao": par_prop_validacao, "teste": par_prop_teste}
df_grupos = df_grupos.sort_values("alunos", ascending=False).reset_index(drop=True)
df_grupos["split"] = None
df_grupos.loc[df_grupos.groupby("sigla_uf").head(1).index, "split"] = "treino"
acumulado = {k: df_grupos.loc[df_grupos["split"] == k, "alunos"].sum() for k in par_metas_split}
for i in df_grupos.index[df_grupos["split"].isna()]:
    destino = max(par_metas_split, key=lambda k: par_metas_split[k] * df_grupos["alunos"].sum() - acumulado[k])
    df_grupos.loc[i, "split"] = destino
    acumulado[destino] += df_grupos.loc[i, "alunos"]
par_split = df_grupos.set_index("nome_mesorregiao")["split"]
df_aluno["split"] = df_aluno["nome_mesorregiao"].map(par_split)
df_municipio["split"] = df_municipio["nome_mesorregiao"].map(par_split)
print("Sem split:", df_aluno["split"].isna().sum(), "alunos |", df_municipio["split"].isna().sum(), "municípios")
df_aluno[df_aluno["ano_alvo"] == par_ano_principal].groupby("split").agg(
    mesorregioes=("nome_mesorregiao", "nunique"), municipios=("id_municipio", "nunique"), ufs=("sigla_uf", "nunique"),
    alunos=("alfabetizado", "size"), pct_alfabetizado=("alfabetizado", lambda s: 100 * s.mean()))

Sem split: 0 alunos | 0 municípios


,mesorregioes,municipios,ufs,alunos,pct_alfabetizado
split,,,,,
teste,33,1204,16,276972,63.135
treino,68,2877,26,1296903,58.972
validacao,34,1436,19,277953,60.237


In [27]:
df_municipio.groupby("split").agg(municipios=("id_municipio", "size"), ufs=("sigla_uf", "nunique"),
                                  pct_atingiu=("atingiu_meta", lambda s: 100 * s.mean()), taxa=("taxa_alfabetizacao", "mean"))

,municipios,ufs,pct_atingiu,taxa
split,,,,
teste,1135,15,55.771,66.156
treino,2738,24,54.456,62.699
validacao,1359,19,48.859,61.997


In [28]:
pd.crosstab(df_aluno.query("ano_alvo == @par_ano_principal")["sigla_uf"], df_aluno.query("ano_alvo == @par_ano_principal")["split"])

split,teste,treino,validacao
sigla_uf,,,
AC,2791,7443,0
AL,5048,20070,6457
AM,6158,40470,0
AP,0,10208,1006
BA,21011,47278,35780
CE,51336,31122,5700
DF,0,22111,0
ES,0,38345,4406
GO,0,54923,15876


**Verificações**

In [29]:
par_proibidas = {"proficiencia", "alfabetizado", "alfabetizado_id", "presenca", "presenca_id", "preenchimento_caderno",
                 "preenchimento_caderno_id", "peso_aluno", "peso_amostral", "taxa_alfabetizacao", "media_portugues",
                 "gap_meta_ano", "gap_ate_2030", "atingiu_meta_ano", "atingiu_meta", "split", "ano", "ano_alvo",
                 "id_aluno", "id_escola", "id_municipio", "nome_mesorregiao", "nome_municipio",
                 *[f"proporcao_aluno_nivel_{i}" for i in range(9)]}
assert not par_proibidas & set(par_features_aluno)
assert not par_proibidas & set(par_features_municipio)
print("Colunas proibidas: nenhuma nas features")

Colunas proibidas: nenhuma nas features


In [30]:
df_ref = pd.concat([f[["ano_alvo", "ano_ref"]].drop_duplicates().assign(fonte=n) for n, f in
                    {"população": df_scr_populacao, "PIB": df_scr_pib, "Censo Escolar": df_scr_censo,
                     "Fundeb municipal": df_scr_fundeb_municipal, "Fundeb estadual": df_scr_fundeb_estadual}.items()])
df_ref = pd.concat([df_ref, pd.DataFrame({"ano_alvo": par_anos_alvo, "ano_ref": par_ano_inep, "fonte": "INEP"}),
                    pd.DataFrame({"ano_alvo": par_anos_alvo, "ano_ref": 2010, "fonte": "Atlas"})])
assert (df_ref["ano_ref"] < df_ref["ano_alvo"]).all()
df_ref.pivot(index="fonte", columns="ano_alvo", values="ano_ref")

ano_alvo,2023,2024,2025
fonte,,,
Atlas,2010,2010,2010
Censo Escolar,2022,2023,2024
Fundeb estadual,2022,2023,2024
Fundeb municipal,2022,2023,2024
INEP,2022,2022,2022
PIB,2021,2022,2023
população,2022,2023,2024


In [31]:
assert df_aluno.groupby("nome_mesorregiao")["split"].nunique().max() == 1
assert df_aluno.groupby("id_municipio")["split"].nunique().max() == 1
print("Nenhum município ou mesorregião em mais de uma partição")

Nenhum município ou mesorregião em mais de uma partição


**Sinal individual de cada feature, medido só no treino**

In [32]:
df_treino_amostra = df_aluno[(df_aluno["ano_alvo"] == par_ano_principal) & (df_aluno["split"] == "treino")].sample(300_000, random_state=par_seed)
par_numericas_aluno = [f for f in par_features_aluno if f not in par_categoricas]
df_auc = auc_univariada(df_treino_amostra, par_numericas_aluno, "alfabetizado")
assert df_auc["forca"].max() < .9
df_auc

,feature,auc,forca,pct_nulos
0,media_portugues_lag,0.626,0.626,2.497
1,taxa_lag_suavizada,0.620,0.620,2.497
2,taxa_lag,0.619,0.619,2.497
3,taxa_lag_vizinhos,0.594,0.594,0.000
4,taxa_lag_uf,0.586,0.586,2.314
5,taxa_presenca_lag,0.582,0.582,28.416
6,taxa_abandono,0.423,0.577,0.002
7,taxa_aprovacao,0.569,0.569,0.002
8,distorcao_idade_serie,0.435,0.565,0.002
9,alunos_por_docente,0.463,0.537,0.000


In [33]:
df_auc_municipio = auc_univariada(df_municipio[df_municipio["split"] == "treino"], [f for f in par_features_municipio if f not in par_categoricas], "atingiu_meta")
df_auc_municipio.head(12)

,feature,auc,forca,pct_nulos
0,taxa_presenca_lag,0.634,0.634,14.536
1,taxa_aprovacao,0.619,0.619,0.037
2,distorcao_idade_serie,0.409,0.591,0.037
3,taxa_abandono,0.419,0.581,0.037
4,escolas_anos_iniciais,0.425,0.575,0.000
5,pct_docente_superior,0.567,0.567,0.037
6,matriculas_anos_iniciais,0.435,0.565,0.000
7,populacao,0.438,0.562,0.000
8,n_avaliados_lag,0.439,0.561,14.536
9,lat,0.557,0.557,0.000


**Decisão:** separação 70/15/15 de alunos por mesorregião, com todas as UFs no treino. As taxas do alvo diferem um pouco (59,0% no treino e 63,1% no teste), e a base municipal herda a mesma separação, ficando em 52/26/22. Nenhuma feature se aproxima do alvo (AUC individual máxima de 0,63). A escolha de features usou a EDA sobre todos os dados: as exclusões por falta de sinal têm esse viés, ainda que pequeno. `alunos_previstos_escola` supõe que a lista de alunos da escola seja conhecida antes da prova; a Gold não informa quando ela é definida, e isso fica registrado como limitação.

### 8. Imputação

In [34]:
df_treino_aluno = df_aluno[(df_aluno["ano_alvo"] == par_ano_principal) & (df_aluno["split"] == "treino")]
df_nulos = pd.DataFrame({"estadual": df_treino_aluno[df_treino_aluno["rede_id"] == par_rede_estadual][par_numericas_aluno].isna().mean() * 100,
                         "municipal": df_treino_aluno[df_treino_aluno["rede_id"] == par_rede_municipal][par_numericas_aluno].isna().mean() * 100})
df_nulos[df_nulos.max(axis=1) > 0].sort_values("municipal", ascending=False)

,estadual,municipal
n_avaliados_lag,61.068,22.867
taxa_presenca_lag,61.068,22.867
taxa_lag_suavizada,13.776,0.555
taxa_lag,13.776,0.555
media_portugues_lag,13.776,0.555
taxa_lag_uf,13.667,0.352
pct_docente_adequado,0.005,0.023
alunos_turma_2ano,0.134,0.019
invest_aluno_fundamental,11.951,0.004
pct_docente_baixa_regularidade,0.000,0.002


In [35]:
df_treino_municipio = df_municipio[df_municipio["split"] == "treino"]
par_numericas_municipio = [f for f in par_features_municipio if f not in par_categoricas]
df_nulos_municipio = df_treino_municipio[par_numericas_municipio].isna().mean().mul(100).rename("pct_nulos").sort_values(ascending=False).to_frame()
df_nulos_municipio[df_nulos_municipio["pct_nulos"] > 0]

,pct_nulos
n_avaliados_lag,14.536
taxa_presenca_lag,14.536
alunos_turma_2ano,0.256
pct_docente_adequado,0.073
invest_aluno_fundamental,0.073
taxa_abandono,0.037
taxa_aprovacao,0.037
pct_docente_superior,0.037
distorcao_idade_serie,0.037
pct_docente_baixa_regularidade,0.037


In [36]:
df_nulos_alvo = df_treino_aluno.assign(lag_ausente=df_treino_aluno["taxa_lag"].isna()).groupby("lag_ausente").agg(
    alunos=("alfabetizado", "size"), ufs=("sigla_uf", "nunique"), pct_alfabetizado=("alfabetizado", lambda s: 100 * s.mean()))
df_nulos_alvo

,alunos,ufs,pct_alfabetizado
lag_ausente,,,
False,1264760,24,58.996
True,32143,11,58.003


**Decisão:** mediana do treino nas numéricas, com indicador de ausência. `taxa_presenca_lag` e `n_avaliados_lag` faltam onde não há microdados de 2023, como SP e AC, então seus indicadores funcionam quase como marcador dessas UFs; o efeito deles é medido nos testes de variáveis dos notebooks 3 e 4. Em 2023 todo o histórico é nulo, e por isso o teste temporal retira a família `historico`.

### 9. Normalização

In [37]:
df_escala = pd.DataFrame({"assimetria_original": df_treino_municipio[[c for c in par_numericas_log if c in df_treino_municipio]].astype(float).skew(),
                          "assimetria_log": np.log1p(df_treino_municipio[[c for c in par_numericas_log if c in df_treino_municipio]].astype(float)).skew(),
                          "minimo": df_treino_municipio[[c for c in par_numericas_log if c in df_treino_municipio]].min()})
assert (df_escala["minimo"] >= 0).all()
df_escala.sort_values("assimetria_original", ascending=False)

,assimetria_original,assimetria_log,minimo
populacao,28.538,0.907,"1,062.000"
n_avaliados_lag,18.918,0.776,9.000
matriculas_anos_iniciais,18.414,0.711,30.000
escolas_anos_iniciais,12.340,0.585,1.000
pib_per_capita,6.872,0.605,"6,757.458"
invest_aluno_fundamental,2.961,0.370,"4,146.410"
renda_pc,0.973,-0.230,107.140
alunos_por_docente,0.716,-0.180,4.765
alunos_por_turma,0.598,-0.559,5.214


In [38]:
df_treino_municipio[[f for f in par_numericas_municipio if f not in par_numericas_log]].astype(float).skew().sort_values(key=abs, ascending=False).rename("assimetria").to_frame()

,assimetria
taxa_abandono,4.277
esforco_meta,-2.289
pct_docente_superior,-1.989
pct_docente_baixa_regularidade,1.948
distorcao_idade_serie,1.885
taxa_aprovacao,-1.342
pct_docente_adequado,-1.084
taxa_lag_uf,-0.651
meta_ano,-0.631
taxa_presenca_lag,-0.520


**Decisão:** log1p nas variáveis muito assimétricas e positivas, seguido de padronização em todas as numéricas. O log reduz a assimetria de `populacao` de 28,5 para 0,9. Investimento igual a zero foi tratado como ausente. Para modelos de árvore, a escala não muda o resultado.

### 10. Encoding

In [39]:
df_cardinalidade = pd.DataFrame({"categorias_treino": df_treino_aluno[par_categoricas].nunique(),
                                 "categorias_total": df_aluno[par_categoricas].nunique(),
                                 "pct_nulos": df_aluno[par_categoricas].isna().mean() * 100})
df_cardinalidade["nao_vistas_no_treino"] = [sorted(set(df_aluno[c].dropna().astype(str)) - set(df_treino_aluno[c].dropna().astype(str))) for c in par_categoricas]
df_cardinalidade

,categorias_treino,categorias_total,pct_nulos,nao_vistas_no_treino
rede,2,2,0.000,[]
sigla_uf,26,26,0.000,[]
nome_regiao,5,5,0.000,[]
capital_uf,2,2,0.000,[]
amazonia_legal,2,2,0.000,[]


**Decisão:** one-hot nas 5 categóricas, de baixa cardinalidade, com `handle_unknown="ignore"`. Todas as categorias aparecem no treino. Não há target encoding, porque `id_municipio`, `id_escola` e mesorregião ficaram fora das features.

### 11. Pré-processador integrado

In [40]:
par_log_aluno = [c for c in par_numericas_log if c in par_features_aluno]
par_num_aluno = [c for c in par_numericas_aluno if c not in par_log_aluno]
par_cat_aluno = [c for c in par_categoricas if c in par_features_aluno]
par_cat_municipio = [c for c in par_categoricas if c in par_features_municipio]
par_log_municipio = [c for c in par_numericas_log if c in par_features_municipio]
par_num_municipio = [c for c in par_numericas_municipio if c not in par_log_municipio]
preprocessador_aluno = construir_preprocessador(par_num_aluno, par_log_aluno, par_cat_aluno)
preprocessador_municipio = construir_preprocessador(par_num_municipio, par_log_municipio, par_cat_municipio)

In [41]:
ajustado_aluno = preprocessador_aluno.fit(df_treino_aluno)
X_treino_aluno = ajustado_aluno.transform(df_treino_aluno)
df_validacao_aluno = df_aluno[(df_aluno["ano_alvo"] == par_ano_principal) & (df_aluno["split"] == "validacao")]
X_validacao_aluno = ajustado_aluno.transform(df_validacao_aluno)
par_nomes_aluno = ajustado_aluno.get_feature_names_out()
print("Treino:", X_treino_aluno.shape, "| Validação:", X_validacao_aluno.shape, "| Nulos:", np.isnan(X_treino_aluno).sum(), np.isnan(X_validacao_aluno).sum())
pd.Series(par_nomes_aluno).str.split("__").str[0].value_counts()

Treino: (1296903, 84) | Validação: (277953, 84) | Nulos: 0 0


cat        37
num        23
ausente    14
log        10
Name: count, dtype: int64

In [42]:
ajustado_municipio = preprocessador_municipio.fit(df_treino_municipio)
X_treino_municipio = ajustado_municipio.transform(df_treino_municipio)
X_teste_municipio = ajustado_municipio.transform(df_municipio[df_municipio["split"] == "teste"])
X_escoragem = ajustado_municipio.transform(df_escoragem)
print("Treino:", X_treino_municipio.shape, "| Teste:", X_teste_municipio.shape, "| Escoragem 2025:", X_escoragem.shape,
      "| Nulos:", np.isnan(X_treino_municipio).sum() + np.isnan(X_teste_municipio).sum() + np.isnan(X_escoragem).sum())

Treino: (2738, 77) | Teste: (1135, 77) | Escoragem 2025: (5352, 77) | Nulos: 0


**O ajuste usa apenas o treino**

In [43]:
mediana_ajustada = pd.Series(ajustado_aluno.named_transformers_["num"]["imputacao"].statistics_, index=par_num_aluno)
pd.testing.assert_series_equal(mediana_ajustada, df_treino_aluno[par_num_aluno].astype(float).median(), check_names=False, check_dtype=False, check_exact=False, atol=1e-3)
df_escala_treino = pd.DataFrame(X_treino_aluno, columns=par_nomes_aluno)
df_escala_validacao = pd.DataFrame(X_validacao_aluno, columns=par_nomes_aluno)
par_escaladas = [c for c in par_nomes_aluno if c.startswith(("num__", "log__"))]
pd.DataFrame({"media_treino": df_escala_treino[par_escaladas].mean(), "dp_treino": df_escala_treino[par_escaladas].std(),
              "media_validacao": df_escala_validacao[par_escaladas].mean()}).sort_values("media_validacao", key=abs, ascending=False).head(10)

,media_treino,dp_treino,media_validacao
log__matriculas_anos_iniciais,0.000,1.000,-0.772
log__populacao,0.000,1.000,-0.736
log__escolas_anos_iniciais,-0.000,1.000,-0.710
log__n_avaliados_lag,-0.000,1.000,-0.626
num__alunos_turma_2ano,0.000,1.000,-0.526
num__pct_docente_baixa_regularidade,-0.000,1.000,0.525
log__alunos_por_turma,-0.000,1.000,-0.520
num__pct_escolas_rurais,-0.000,1.000,0.510
num__pct_esgoto_rede,0.000,1.000,-0.436
num__idhm,-0.000,1.000,-0.427


In [44]:
pd.Series(par_nomes_aluno[[c.startswith("ausente__") for c in par_nomes_aluno]], name="indicadores_de_ausencia").to_frame()

,indicadores_de_ausencia
0,ausente__missingindicator_taxa_lag
1,ausente__missingindicator_taxa_lag_suavizada
2,ausente__missingindicator_media_portugues_lag
3,ausente__missingindicator_taxa_presenca_lag
4,ausente__missingindicator_taxa_lag_uf
5,ausente__missingindicator_alunos_turma_2ano
6,ausente__missingindicator_pct_docente_superior
7,ausente__missingindicator_pct_docente_adequado
8,ausente__missingindicator_pct_docente_baixa_re...
9,ausente__missingindicator_distorcao_idade_serie


**Uso na modelagem:** carregar o `.joblib` não ajustado e colocá-lo como primeira etapa do `Pipeline`, para que cada dobra ajuste imputação, escala e encoding só no próprio treino. O pré-processador ajustado neste notebook serve apenas de verificação.

### 12. Salvar artefatos

In [45]:
par_saida.mkdir(parents=True, exist_ok=True)
df_aluno.drop(columns=["nome_municipio", "nome_uf"], errors="ignore").to_parquet(par_saida / "base_aluno.parquet", index=False)
df_municipio.to_parquet(par_saida / "base_municipio.parquet", index=False)
df_escoragem.to_parquet(par_saida / "base_municipio_escoragem_2025.parquet", index=False)
df_catalogo.to_csv(par_saida / "catalogo_features.csv", index=False)
df_excluidas.to_csv(par_saida / "features_excluidas.csv", index=False)
joblib.dump(construir_preprocessador(par_num_aluno, par_log_aluno, par_cat_aluno), par_saida / "preprocessador_aluno.joblib")
joblib.dump(construir_preprocessador(par_num_municipio, par_log_municipio, par_cat_municipio), par_saida / "preprocessador_municipio.joblib")
pd.DataFrame([{"arquivo": p.name, "mb": p.stat().st_size / 1e6} for p in sorted(par_saida.iterdir())])

,arquivo,mb
0,base_aluno.parquet,5.117
1,base_municipio.parquet,0.687
2,base_municipio_escoragem_2025.parquet,0.680
3,catalogo_features.csv,0.006
4,features_excluidas.csv,0.001
5,preprocessador_aluno.joblib,0.002
6,preprocessador_municipio.joblib,0.002


In [46]:
df_conferencia = pd.read_parquet(par_saida / "base_aluno.parquet", columns=["ano_alvo", "split", "alfabetizado"])
assert len(df_conferencia) == len(df_aluno)
novo = joblib.load(par_saida / "preprocessador_municipio.joblib").fit(df_treino_municipio)
assert np.allclose(novo.transform(df_escoragem), X_escoragem)
df_conferencia.groupby(["ano_alvo", "split"]).size().unstack()

split,teste,treino,validacao
ano_alvo,,,
2023,282649,974769,245391
2024,276972,1296903,277953
